# Quick introduction to Docker

**Docker** is a containerization platform: it lets you package an application (e.g., a FastAPI API) together with its dependencies into a **container image**, and then run that image as a **container** on any machine with a compatible runtime. The practical goal is to eliminate “works on my machine” by standardizing the runtime environment. 

## 1) Why containerization exists (the problem Docker solves)

### The pain

Runtime differences cause deployment failures: OS libs, Python version, system packages, env vars, ports, paths, etc.

### The Docker idea

* Build an **image** that contains your app + dependencies.
* Run it as a **container** (isolated process) anywhere Docker can run.

**Mental model:** **Image = blueprint**, **Container = running instance.** 

## 2) Core Docker components

### Image

* Read-only template used to create containers. 

### Container

* A running instance of an image (with a writable layer). 

### Dockerfile

* The build recipe for an image(base image, copy files, install deps, set command, etc. 

### Registry

* Storage for images (pull/push). Common examples(Docker Hub, GHCR, ACR. 

### Docker Engine / daemon

* The service that builds images and runs containers (local or remote). 


## 3) Key container features

### Isolation (not full virtualization)

Containers share the host OS kernel and isolate processes/filesystems/network namespaces, which is why they start fast and are lightweight compared to VMs. 

### Layered builds + caching

Each Dockerfile instruction produces a layer. Docker can reuse unchanged layers to speed up rebuilds. 

### Port mapping

Publish container ports to the host:

* `-p 8000:8000` → host port 8000 → container port 8000
  Docker documents this as “published ports / --publish (-p)”. 

### Environment variables

Inject config without changing code (e.g., `-e ENV=prod`). (This is core runtime behavior; commonly taught alongside `docker run`.)

### Volumes (persistence + dev mounts)

Containers are ephemeral; volumes persist data.

* **Volumes**(managed by Docker (recommended for persistence). 
* **Bind mounts**: map a host folder into the container (great for live dev). 

### Networking

Docker provides networks; by default Docker creates a bridge network, and you can create your own for app-to-app communication. 


## 4) Installing Docker (practical setup)

### Typical setup

* **Docker Desktop** for Mac/Windows/Linux (includes engine + tooling; UI optional). 
* On Windows, note Docker Desktop licensing guidance and install steps live in Docker docs. 

### Verify

* `docker --version`
* `docker run hello-world` → prints “Hello from Docker!” if it works. 


# Practical workflow: containerizing a FastAPI-style API

## A minimal Dockerfile pattern (FastAPI-friendly)

Key idea: copy `requirements.txt` early for cache efficiency, then copy your source.

```dockerfile
FROM python:3.13
WORKDIR /usr/local/app

COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt

COPY . .
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

This structure matches Docker’s own “writing a Dockerfile” guidance for Python apps and common Dockerfile primitives (`FROM`, `WORKDIR`, `RUN`, `COPY`, `CMD`). 

## Build + run

```bash
docker build -t myapi:dev .
docker run --rm -p 8000:8000 myapi:dev
```

Port publishing (`-p/--publish`) is the standard way to expose the API outside the container. 

## Dev-mode option (live code)

Use a bind mount so code edits reflect in-container:

```bash
docker run --rm -p 8000:8000 -v "$(pwd)":/usr/local/app myapi:dev
```

Bind mounts map a host directory directly into the container. 


# Common commands cheat sheet

* **Images**

  * `docker build -t name:tag .` (build an image)
  * `docker images` (list local images)
* **Containers**

  * `docker run ... image` (create + start)
  * `docker ps` / `docker ps -a` (running / all)
  * `docker logs -f <container>` (follow logs)
  * `docker exec -it <container> bash` (shell inside)
  * `docker stop <container>` / `docker rm <container>` (stop/remove)
* **Cleanup**

  * `docker system prune` (clear unused resources—use carefully)
* **Volumes**

  * `docker volume create myvol` (create) 
  * `docker volume ls` / `docker volume rm ...`
* **Networks**

  * `docker network create mynet` (create a network) 


# When to introduce Docker Compose (multi-container)

The moment you add a DB/cache/queue, you’ll want **Compose**:

* One `compose.yml` defines **services + networks + volumes**
* Bring everything up/down with a command

Docker’s “multi-container applications” docs describe Compose as the standard approach. 

# Useful Links
- [Docker Desktop](https://docs.docker.com/desktop/)
- [Docker Image vs Container](https://aws.amazon.com/compare/the-difference-between-docker-images-and-containers/) 
- [Writing a Dockerfile](https://docs.docker.com/get-started/docker-concepts/building-images/writing-a-dockerfile/) 
- [What's the Difference Between Docker and a VM?](https://aws.amazon.com/compare/the-difference-between-docker-vm/) 
- [Volumes](https://docs.docker.com/engine/storage/volumes/)
- [Data Persistence with Docker Bind Mounts](https://medium.com/my-docker-journal/data-persistence-with-docker-bind-mounts-5036d2be287e)
- [Multi-container applications](https://docs.docker.com/get-started/docker-concepts/running-containers/multi-container-applications/) ""

> Content created by [**Carlos Cruz-Maldonado**](https://www.linkedin.com/in/carloscruzmaldonado/).  
> I am available to answer any questions or provide further assistance.   
> Feel free to reach out to me at any time.